### Tables of units and blades' map.

In [13]:
AMPSUB = {
    0    : 1.0,    # no unit defined.
    "mA" : 1e-3,   # mili
    "uA" : 1e-6,   # micro
    "nA" : 1e-9,   # nano
    "pA" : 1e-12,  # pico
    "fA" : 1e-15,  # femto
    "aA" : 1e-18,  # atto
}

# The XBPM beamlines.
BEAMLINENAME = {
    "CAT": "Cateretê",
    "CNB": "Carnaúba",
    "MGN": "Mogno",
    "MNC": "Manacá",
}

BLADEMAP = {
    "MNC": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MNC1": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MNC2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "CAT":  {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    # "CAT1": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},

    # ## To be checked: ## #
    # "CAT2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "CNB": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    # "CNB1": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    # "CNB2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "MGN": {"TO": 'C', "TI": 'A', "BI": 'D', "BO": 'B'},
    "MGN1": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "MGN2": {"TO": 'B', "TI": 'A', "BI": 'C', "BO": 'D'},
    "SIM":  {"TO": 'A', "TI": 'B', "BI": 'C', "BO": 'D'},
}

FILE_EXTENSION = ".pickle"    # Data file type.

### Procedures for 2025-06-11

#### Set of functions to read data from files and restructure them for each beamline, in columns for each blade's value, the undulator gap and the SR current.

In [7]:
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pickle  # noqa: S403


In [ ]:
# Functions for opening, rearranging and plotting data.

def data_from_file(wdir):
    """Read data from pickle files."""
    allfiles = os.listdir(wdir)
    picklefiles = [pf for pf in allfiles if pf.endswith("pickle")]
    sfiles = sorted(picklefiles, key=lambda name: lastfield(name, '_'))

    rawdata = list()
    for file in sfiles:
        with open(wdir + "/" + file, 'rb') as df:
            rawdata.append(pickle.load(df))  # noqa: S301

    return rawdata


def lastfield(name, fld=' '):  # noqa: D103
    return name.split(fld)[-1]


def read_rawdata(rawdata):
    """."""
    data = dict()
    blk = 0
    for nline in range(6):
        beamline = rawdata[blk][0][nline]['name']
        data[beamline] = dict()

    for blk in range(len(rawdata)):
        for nline, bline in enumerate(data.keys()):
            data[bline][blk] = dict()
            # For each block.
            data[bline][blk]['data'] = deepcopy(rawdata[blk][0][nline])
            data[bline][blk]['info'] = deepcopy(rawdata[blk][1])

    return data


def data_structure_average(rdata):
    """Average over blades' values and simplify data structure."""
    data = dict()
    for bline in rdata.keys():
        for blk in range(len(rdata[bline].keys())):
            for bld in ["A", "B", "C", "D"]:
                blade = f"{bld}_val"
                av = average_blade(rdata[bline][blk]['data'][blade])
                rdata[bline][blk]['data'][blade] = av
                del rdata[bline][blk]['data'][f"{bld}_range"]
            del rdata[bline][blk]['data']["name"]
            del rdata[bline][blk]['data']["prefix"]

            rdata[bline][blk]['data']["current"] = \
                np.round(rdata[bline][blk]['info']["current"], decimals=-1)
            del rdata[bline][blk]['info']["current"]

            bl = bline[:3].lower()   # Gap info key.
            if bl in rdata[bline][blk]['info'].keys():
                rdata[bline][blk]['data']['gap'] = \
                    np.round(rdata[bline][blk]['info'][bl], decimals=0)

            del rdata[bline][blk]["info"]
            rdata[bline][blk] = rdata[bline][blk]["data"]

        dt = dict()
        for key, rd in rdata[bline].items():
            dt[key] = rd
        data[bline] = dt
    return data


def average_blade(blade):
    """Function for averaging over each blade's measurement."""
    bld = []
    for val in blade:
        bld.append(val[0] * AMPSUB[val[1]])
    return np.array([np.average(bld), np.std(bld)])


def pandas_data_frame(data):
    """Data to pandas' data frame."""
    pdata = dict()
    for key, val in data.items():
        pdata[key] = pd.DataFrame(val)
    return pdata


def blades_data_array(pdata, beamline):
    """Divide data into arrays for each blade."""
    blades = {
        "A_val" : [],
        "B_val" : [],
        "C_val" : [],
        "D_val" : [],
    }

    for ip, p in enumerate(pdata[beamline]):
        crr = np.round(pdata[beamline][p]["current"], decimals=-1)

        try:
            if beamline in ["CAT", "CNB"]:
                gap = np.round(pdata[beamline][p]["gap"], decimals=0)
            elif beamline in ["MNC1", "MNC2"]:
                gap = pdata[beamline][p]["gap"]
            else:
                gap = None  # pdata[beamline][p]["gap"]
        except Exception as err:
            print(f" WARNING: beamline {beamline}, data # {ip}:"
                  f"\t Exception when trying to define {err}")
            gap = None

        # print(f">>> (BLADES DATA RRAY) gap {beamline} = {gap}")

        for bl in blades.keys():
            blval = pdata[beamline][p][bl]
            # print(f" {cr} ({type(cr)}) : {bl} → {blval}")
            blades[bl].append([crr, gap, blval[0], blval[1]])

    for key, val in blades.items():
        blades[key] = np.array(val)

    return blades


def blades_plot(blades, ax0, ax1, beamline):
    """Plot data."""
    # Plot markers for each blade.
    markers = {'A_val' : 'o',
               'B_val' : '^',
               'C_val' : 's',
               'D_val' : '*'}

    for key, val in blades.items():
        x = val[:, 0]
        y = val[:, 2]
        s = val[:, 3]

        # beamlines have different amperimeters.
        if beamline in ["CAT", "CNB"]:
            ylabel = u"$I$ [nA]"
            y *= 1e9     # Current in nA.
        else:
            ylabel = "counts"

        mrk = markers[key]
        mask = val[:, 2] >= -5e-3
        if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
            # Beamlines with undulator source.
            gaps = np.unique(val[:, 1])
            # Divide in groups by undulator's gap.
            for gap in gaps:
                mask_gap = val[:, 1] == gap
                gmask = mask & mask_gap
                ax0.errorbar(x[gmask], y[gmask], s[gmask],
                            fmt=f'{mrk}-', label=key)
                ax1.errorbar(x[gmask], y[gmask], s[gmask],
                            fmt=f'{mrk}-', label=f"{key} @ gap {gap}")

        else:
            # Beamline with dipole source.
            ax0.errorbar(x[mask], y[mask], s[mask],
                         fmt=f'{mrk}-', label=key)

    title_curr = u"XBPM $\\times$ SR currents @"
    ax0.set_title(f"{title_curr} {beamline}")
    ax0.set_xlabel("SR current [mA]")
    ax0.set_ylabel(ylabel)
    ax0.legend()
    ax0.grid()

    if ax1 is not None:
        title_gap = u"XBPM $\\times$ undulator gaps @"
        ax1.set_title(f"{title_gap} {beamline}")
        ax1.set_xlabel("SR current [mA]")
        ax1.set_ylabel(ylabel)
        ax1.legend()
        ax1.grid()


In [9]:
%matplotlib qt5

## Main.

In [79]:
def main(wdir, outdir):
    """."""
    # List with data read from working directory.
    rawdata = data_from_file(wdir)

    # Extract data from read files and compose a dict
    # relative to each beam line.
    rdata   = read_rawdata(rawdata)

    # Average over blades' measurements and
    # simplify data structure dictionary.
    data    = data_structure_average(rdata)

    # Export data to a pandas data frame.
    pdata   = pandas_data_frame(data)

    for beamline in pdata.keys():

        # Divide pdata into arrays for each blade.
        blades = blades_data_array(pdata, beamline=beamline)

        if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
            fig, (axi, axg) = plt.subplots(1, 2, figsize=(20, 9))
            blades_plot(blades, axi, axg, beamline)
        else:
            fig, axi = plt.subplots(1, 1, figsize=(10, 6))
            blades_plot(blades, axi, None, beamline)

        fig.tight_layout()
        fig.savefig(f"{outdir}/XBPM_{beamline}_SR_current.png")

    plt.show()


basedir = "/home/arnaldo.filho/XBPM/medidas/results_2025/"

# Gap and current variation.
outdir = basedir + "Gaps_20250616/"
wdir = outdir + "2025-06-16"

outdir = basedir + "MNC_1_20250623/"
wdir = outdir + "2025-06-23"

# wdir = "MNC_1_20250729/subsec_09SA"
# wdir = "MNC_1_20250623/2025-06-23"

# wdir = basedir + "Gaps_20250616/2025-06-16"

# wdir = basedir + "MNC_1_20250623/2025-06-23"

if __name__ == "__main__":
    main(wdir, outdir)

KeyError: 0

In [68]:
wdir = ("/home/arnaldo.filho/XBPM/medidas/results_2025/" +
        "Gaps_20250616/2025-06-16")

rawdata = data_from_file(wdir)
rdata = read_rawdata(rawdata)

for rd in rdata:
    print(rdata[rd])


{0: {'data': {'name': 'CNB', 'prefix': 'CNB:FE:PICO01', 'A_val': [(-28.222798989334386, 'fA'), (-16.391276763628657, 'fA'), (-26.822089403580154, 'fA'), (-18.179415831325215, 'fA'), (-27.50754403988212, 'fA'), (-17.46416088187295, 'fA'), (-21.964311405363485, 'fA'), (-15.228986623735778, 'fA'), (-20.861625279903, 'fA'), (-16.063451296118718, 'fA')], 'B_val': [(27.942658088623077, 'fA'), (31.23283221135621, 'fA'), (22.697448914398183, 'fA'), (30.040741758313033, 'fA'), (25.84457441893312, 'fA'), (31.948090548940268, 'fA'), (24.55711313822679, 'fA'), (24.843215456820875, 'fA'), (19.025803339473708, 'fA'), (26.98898369330946, 'fA')], 'C_val': [(-22.47095061023649, 'fA'), (-7.5697903010435965, 'fA'), (-16.689300223922398, 'fA'), (-7.4803825853291155, 'fA'), (-9.208917638593283, 'fA'), (-6.228685576754704, 'fA'), (-18.060207802460432, 'fA'), (-7.838010907088195, 'fA'), (-18.03040579524424, 'fA'), (-8.404254973426537, 'fA')], 'D_val': [(-360.60809893712076, 'pA'), (-366.56855328942584, 'pA')

In [134]:
wdir = ("/home/arnaldo.filho/XBPM/medidas/results_2025/" +
        "MNC_1_20250623/2025-06-23")

rawdata = data_from_file(wdir)

rdata = { key : {} for key in rawdata[0][0].keys()}
for rd in rawdata:
    # print(f" RD 1 current = {np.round(rd[1]['current'], decimals=-1)}")

    # print(f" RD 0 : {rd[0].keys()}\n")

    for key, val in rd[0].items():
        current = np.round(rd[1]["current"], decimals=-1)
        rdata[key][current] = {}
        rdata[key][current]["gap"] = round(rd[1]['cnb'])

        for blade in ["A_val", "B_val", "C_val", "D_val"]:
            rdata[key][current][blade] = average_blade(val[blade])

pdata = pandas_data_frame(rdata)
# for beamline in pdata.keys():

#     # Divide pdata into arrays for each blade.
#     blades = blades_data_array(pdata, beamline=beamline)

    # if beamline in ["CAT", "CNB", "MNC1", "MNC2"]:
    #     fig, (axi, axg) = plt.subplots(1, 2, figsize=(20, 9))
    #     blades_plot(blades, axi, axg, beamline)
    # else:
    #     fig, axi = plt.subplots(1, 1, figsize=(10, 6))
    #     blades_plot(blades, axi, None, beamline)

    # fig.tight_layout()
# rawdata[0][0].keys()
pdata['CNB'].T

,gap,A_val,B_val,C_val,D_val
80.0,11,"[4.207967276670388e-06, 4.371960029404878e-09]","[4.564057326206239e-06, 3.6951092054409474e-09]","[3.7222862147245906e-06, 4.953277122665135e-09]","[3.971672549596406e-06, 4.253193949007398e-09]"
100.0,11,"[5.191422906136722e-06, 1.2807529515843308e-08]","[5.631871681543999e-06, 7.3798509675712404e-09]","[4.591449715007911e-06, 1.1104329839783808e-08]","[4.9011031023837855e-06, 8.447209650823668e-09]"
120.0,11,"[6.1809692397218894e-06, 2.80959492985688e-10]","[6.731634221068816e-06, 2.340438255475132e-10]","[5.414689485405688e-06, 2.5202100237496177e-10]","[5.833573914060252e-06, 1.813200409727553e-10]"
140.0,11,"[7.12063406353991e-06, 3.454369623457906e-10]","[7.749831820547114e-06, 2.751066779249957e-10]","[6.229784958122764e-06, 3.296060748430288e-10]","[6.711921241731034e-06, 2.8061592403300436e-10]"
160.0,11,"[8.119122412608704e-06, 4.053772560208827e-10]","[8.834410982672125e-06, 3.5429300443176814e-10]","[7.09910877958464e-06, 4.148652904270811e-10]","[7.650706902495585e-06, 4.95569600596404e-10]"
180.0,11,"[9.098014834307832e-06, 4.0964038159245547e-10]","[9.893540391203714e-06, 4.1484525919273634e-10]","[7.947960966703249e-06, 3.0361250388212355e-10]","[8.564110248698854e-06, 4.16807401119633e-10]"
200.0,11,"[1.0064239450002788e-05, 4.816627697905546e-10]","[1.0935186674032593e-05, 4.389605207173421e-10]","[8.781183350947685e-06, 6.225743083502328e-10]","[9.458471413381631e-06, 6.607454972138017e-10]"
